# DICOM coordinates and gamma, illustrated

This page shows how PyMedPhys maps a DICOM RT Dose grid to patient coordinates, what the previous conversion got wrong, how gamma treats the order of its input axes, and how to plot gamma on the reordered grids. The code that draws each figure also checks the claim the figure illustrates. The [DICOM coordinate validation note](https://docs.pymedphys.com/en/latest/contrib/info/dicom-coordinate-validation.html) records the derivation, the test evidence, and the remaining limitations.

Every dataset is generated in memory, so the page needs no downloads. The previous behaviour comes from a copy of the conversion code in PyMedPhys 0.41.0, included in the setup section.

In summary:

- For all eight supported orientations, every voxel returned by `pymedphys.dicom.zyx_and_dose_from_dataset` sits exactly where the DICOM standard places it (section 2).
- The previous conversion translated grids along reversed axes by twice their centre coordinate and transposed decubitus grids. Crops, shifts, and comparisons between grids of different extent were therefore misregistered (section 3).
- Gamma reverses descending evaluation axes internally, and returns its result on the reference grid, in the reference order (section 4). Plot gamma against the reference axes, or restore the stored order explicitly (section 5).
- Gamma is at least as fast as before, and faster with the default interpolator (section 6).
- Section 7 shows how to check whether an earlier comparison was affected.

## Setup

Numba is limited to two threads so that the examples suit documentation build hosts. The colours are used consistently: blue for the current conversion, orange for the previous one, and black for positions calculated independently from the DICOM standard.

In [ ]:
import copy
import time

import numba
import numpy as np
import pydicom
import scipy.ndimage
from matplotlib import colors
from matplotlib import pyplot as plt

import pymedphys
from pymedphys._dicom.coords import coords_in_datasets_are_equal
from pymedphys._gamma.implementation.shell import (
    _grid_distance_bounds,
    _prepare_evaluation_grid,
)
from pymedphys._interp.interp import interp_linear_1d, interp_linear_3d

numba.set_num_threads(min(2, numba.get_num_threads()))

CURRENT, PREVIOUS, TRUTH = "#2a78d6", "#eb6834", "#0b0b0b"
SECONDARY, MUTED = "#52514e", "#898781"
DOSE_CMAP = colors.LinearSegmentedColormap.from_list(
    "dose", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
# Gamma diverges at 1: blue passes, red fails, grey sits on the threshold.
GAMMA_CMAP = colors.LinearSegmentedColormap.from_list(
    "gamma", ["#2a78d6", "#f0efec", "#e34948"]
)

plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": SECONDARY,
        "axes.titlecolor": TRUTH,
        "axes.titlesize": 10,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": SECONDARY,
        "ytick.labelcolor": SECONDARY,
        "text.color": TRUTH,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.8,
        "lines.linewidth": 2,
        "legend.frameon": False,
        "font.size": 9,
        "figure.dpi": 110,
    }
)

The helpers below build RT Dose datasets in memory and calculate where DICOM places every voxel. `voxel_positions` evaluates the standard's definition (section 1) for every voxel without calling PyMedPhys, so it serves as an independent reference. Unless stated otherwise, every stored voxel holds a distinct value, its storage index, so each returned dose value identifies the stored voxel it came from.

In [ ]:
ORIENTATIONS = {
    "HFS": (1, 0, 0, 0, 1, 0),
    "HFP": (-1, 0, 0, 0, -1, 0),
    "FFS": (-1, 0, 0, 0, 1, 0),
    "FFP": (1, 0, 0, 0, -1, 0),
    "HFDL": (0, -1, 0, 1, 0, 0),
    "HFDR": (0, 1, 0, -1, 0, 0),
    "FFDL": (0, 1, 0, 1, 0, 0),
    "FFDR": (0, -1, 0, -1, 0, 0),
}
DOSE_GRID_SCALING = 1e-4  # Gy per stored unit


def make_rtdose(orientation, position, pixels, pixel_spacing, slice_offsets=None):
    """Build an RT Dose dataset from stored pixels in (slice, row, column) order."""
    pixels = np.asarray(pixels)
    slices, rows, columns = pixels.shape
    ds = pydicom.Dataset()
    ds.file_meta = pydicom.dataset.FileMetaDataset()
    ds.file_meta.TransferSyntaxUID = pydicom.uid.ImplicitVRLittleEndian
    ds.Modality = "RTDOSE"
    ds.ImagePositionPatient = [round(float(value), 4) for value in position]
    ds.ImageOrientationPatient = list(ORIENTATIONS[orientation])
    ds.PixelSpacing = [round(float(value), 4) for value in pixel_spacing]
    ds.Rows, ds.Columns, ds.NumberOfFrames = rows, columns, slices
    if slice_offsets is not None:
        ds.GridFrameOffsetVector = [round(float(value), 4) for value in slice_offsets]
    ds.BitsAllocated = ds.BitsStored = 32
    ds.HighBit = 31
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.DoseUnits, ds.DoseType, ds.DoseSummationType = "GY", "PHYSICAL", "PLAN"
    ds.DoseGridScaling = DOSE_GRID_SCALING
    ds.PixelData = np.ascontiguousarray(pixels, dtype="<u4").tobytes()
    return ds


def voxel_positions(ds):
    """Patient (x, y, z) of every stored voxel, indexed [slice, row, column, xyz].

    Evaluates DICOM PS3.3 C.7.6.2.1.1 and C.8.8.3.2 term by term.
    """
    S = np.array(ds.ImagePositionPatient, dtype=float)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)
    r, c = iop[:3], iop[3:]
    row_spacing, column_spacing = (float(value) for value in ds.PixelSpacing)
    offsets = getattr(ds, "GridFrameOffsetVector", None)
    g = np.atleast_1d(np.array(offsets if offsets else [0.0], dtype=float))
    if g[0] != 0:  # absolute z coordinates rather than offsets
        g = g - S[2]
    k, i, j = np.meshgrid(
        np.arange(g.size), np.arange(ds.Rows), np.arange(ds.Columns), indexing="ij"
    )
    return (
        S
        + (j * column_spacing)[..., None] * r
        + (i * row_spacing)[..., None] * c
        + g[k][..., None] * np.cross(r, c)
    )


def returned_positions(ds, axes_zyx, dose_zyx):
    """Coordinates given to each stored voxel by a conversion, as [voxel, xyz].

    Returns None when the conversion gave axes that do not match the dose shape.
    """
    z, y, x = axes_zyx
    if np.shape(dose_zyx) != (len(z), len(y), len(x)):
        return None
    Z, Y, X = np.meshgrid(z, y, x, indexing="ij")
    stored_index = np.rint(np.ravel(dose_zyx) / DOSE_GRID_SCALING).astype(int)
    positions = np.empty((stored_index.size, 3))
    positions[stored_index] = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=-1)
    return positions


def placement_error(ds, axes_zyx, dose_zyx):
    """Largest distance (mm) between each returned voxel and its DICOM position."""
    positions = returned_positions(ds, axes_zyx, dose_zyx)
    if positions is None:
        return np.nan
    true = voxel_positions(ds).reshape(-1, 3)
    return np.max(np.linalg.norm(positions - true, axis=-1))

The previous conversion is reproduced below from PyMedPhys 0.41.0, whose coordinate and gamma code behaves identically to `main` before this change. The function bodies are unchanged; the three functions called below have gained a `legacy_` prefix, and the docstrings are omitted.

In [ ]:
# Reproduced from pymedphys._dicom.coords and pymedphys._dicom.dose in
# PyMedPhys 0.41.0, for comparison only.


def _orientation_is_head_first(orientation_vector, is_decubitus):
    if is_decubitus:
        return np.abs(np.sum(orientation_vector)) != 2

    return np.abs(np.sum(orientation_vector)) == 2


def legacy_xyz_axes_from_dataset(ds, coord_system="DICOM"):
    position = np.array(ds.ImagePositionPatient)
    orientation = np.array(ds.ImageOrientationPatient)

    if not (
        np.array_equal(np.abs(orientation), np.array([1, 0, 0, 0, 1, 0]))
        or np.array_equal(np.abs(orientation), np.array([0, 1, 0, 1, 0, 0]))
    ):
        raise ValueError(
            "Dose grid orientation is not supported. Dose "
            "grid slices must be aligned along the "
            "superoinferior axis of patient."
        )

    is_decubitus = orientation[0] == 0
    is_head_first = _orientation_is_head_first(orientation, is_decubitus)

    row_spacing = float(ds.PixelSpacing[0])
    column_spacing = float(ds.PixelSpacing[1])

    row_range = np.array([row_spacing * i for i in range(ds.Rows)])
    col_range = np.array([column_spacing * i for i in range(ds.Columns)])

    if is_decubitus:
        x_dicom_fixed = orientation[1] * position[1] + col_range
        y_dicom_fixed = orientation[3] * position[0] + row_range
    else:
        x_dicom_fixed = orientation[0] * position[0] + col_range
        y_dicom_fixed = orientation[4] * position[1] + row_range

    if is_head_first:
        z_dicom_fixed = position[2] + np.array(ds.GridFrameOffsetVector)
    else:
        z_dicom_fixed = -position[2] + np.array(ds.GridFrameOffsetVector)

    if coord_system.upper() in ("FIXED", "IEC FIXED", "F"):
        x = x_dicom_fixed
        y = z_dicom_fixed
        z = -np.flip(y_dicom_fixed)

    elif coord_system.upper() in ("DICOM", "D", "PATIENT", "IEC PATIENT", "P"):
        if orientation[0] == 1:
            x = x_dicom_fixed
        elif orientation[0] == -1:
            x = np.flip(x_dicom_fixed)
        elif orientation[1] == 1:
            y_d = x_dicom_fixed
        elif orientation[1] == -1:
            y_d = np.flip(x_dicom_fixed)

        if orientation[4] == 1:
            y_d = y_dicom_fixed
        elif orientation[4] == -1:
            y_d = np.flip(y_dicom_fixed)
        elif orientation[3] == 1:
            x = y_dicom_fixed
        elif orientation[3] == -1:
            x = np.flip(y_dicom_fixed)

        if not is_head_first:
            z_d = np.flip(z_dicom_fixed)
        else:
            z_d = z_dicom_fixed

        if coord_system.upper() in ("DICOM", "D"):
            y = y_d
            z = z_d
        elif coord_system.upper() in ("PATIENT", "IEC PATIENT", "P"):
            y = z_d
            z = -np.flip(y_d)

    return (x, y, z)


def legacy_zyx_and_dose_from_dataset(dataset):
    x, y, z = legacy_xyz_axes_from_dataset(dataset)
    coords = (z, y, x)
    dose = dataset.pixel_array * dataset.DoseGridScaling

    return coords, dose


def legacy_coords_in_datasets_are_equal(datasets):
    # Quick shape (sanity) check
    if not all(
        ds.pixel_array.shape == datasets[0].pixel_array.shape for ds in datasets
    ):
        return False

    # Full coord check:
    all_concat_axes = [
        np.concatenate(legacy_xyz_axes_from_dataset(ds)) for ds in datasets
    ]

    return all(np.allclose(a, all_concat_axes[0]) for a in all_concat_axes)

## 1. From stored pixels to patient coordinates

DICOM defines the patient position of the stored voxel in slice $k$, row $i$, and column $j$ of `pixel_array[k, i, j]` as

$$
\mathbf{P}_{kij} = \mathbf{S} + j\,\Delta_c\,\hat{\mathbf{r}} + i\,\Delta_r\,\hat{\mathbf{c}} + g_k\,\hat{\mathbf{n}}, \qquad \hat{\mathbf{n}} = \hat{\mathbf{r}} \times \hat{\mathbf{c}},
$$

where:

- $\mathbf{S}$ is `ImagePositionPatient`, the centre of the first stored voxel;
- $\hat{\mathbf{r}}$ and $\hat{\mathbf{c}}$ are the first and second triplets of `ImageOrientationPatient`, the directions of increasing column index $j$ and increasing row index $i$;
- $\Delta_r$ is `PixelSpacing[0]`, the spacing between rows, and $\Delta_c$ is `PixelSpacing[1]`, the spacing between columns;
- $g_k$ is `GridFrameOffsetVector[k]`, the offset of slice $k$ along $\hat{\mathbf{n}}$. A vector whose first element is not zero holds absolute z coordinates instead, which the standard permits only for `ImageOrientationPatient` $= (1, 0, 0, 0, 1, 0)$; subtracting $S_z$ converts it to offsets.

Equivalently, as a homogeneous matrix,

$$
\begin{bmatrix} x \\ y \\ z \\ 1 \end{bmatrix} =
\begin{bmatrix}
\Delta_c r_x & \Delta_r c_x & n_x & S_x \\
\Delta_c r_y & \Delta_r c_y & n_y & S_y \\
\Delta_c r_z & \Delta_r c_z & n_z & S_z \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix} j \\ i \\ g_k \\ 1 \end{bmatrix}.
$$

Changing the orientation changes the direction cosines, never the sign of $\mathbf{S}$. The figure below draws a stored grid of four rows and five columns in each of the eight supported orientations, in the transverse plane as it is usually displayed: patient left to the right, posterior downwards.

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(12.5, 6.8), sharex=True, sharey=True, layout="constrained")
spacing, rows, columns = 6.0, 4, 5
for ax, (name, iop) in zip(axs.flat, ORIENTATIONS.items()):
    r, c = np.array(iop[:3], dtype=float), np.array(iop[3:], dtype=float)
    # Choose S so that the grid is centred on the origin.
    S = -spacing * ((columns - 1) / 2 * r + (rows - 1) / 2 * c)
    j, i = np.meshgrid(np.arange(columns), np.arange(rows))
    centres = S + spacing * (j[..., None] * r + i[..., None] * c)
    ax.scatter(centres[..., 0], centres[..., 1], s=16, color=MUTED, zorder=2)
    ax.scatter(S[0], S[1], marker="*", s=220, color=TRUTH, zorder=4)
    for vector, label, count in ((r, "j: columns", columns), (c, "i: rows", rows)):
        end = S[:2] + (count - 1) * spacing * vector[:2]
        ax.annotate(
            "", xy=end, xytext=S[:2],
            arrowprops={"arrowstyle": "-|>", "color": TRUTH, "lw": 1.6, "shrinkA": 6, "shrinkB": 0},
            zorder=3,
        )
        horizontal = {1: "left", -1: "right", 0: "center"}[int(vector[0])]
        vertical = {1: "top", -1: "bottom", 0: "center"}[int(vector[1])]
        ax.text(*(end + 0.45 * spacing * vector[:2]), label, ha=horizontal, va=vertical, fontsize=8.5)
    n = np.cross(r, c)
    ax.text(
        0.03, 0.03, f"slices run {'+z, towards the head' if n[2] > 0 else '−z, towards the feet'}",
        transform=ax.transAxes, color=SECONDARY, fontsize=8.5,
    )
    ax.set_title(f"{name}   {list(iop)}")
    ax.set_aspect("equal")
    ax.set_xlim(-30, 30)
    ax.set_ylim(-26, 26)
axs[0, 0].invert_yaxis()
for ax in axs[1]:
    ax.set_xlabel("x (mm), towards patient left")
for ax in axs[:, 0]:
    ax.set_ylabel("y (mm), towards posterior")
fig.suptitle("Stored voxel centres in each orientation; ★ marks the first stored voxel, S")
plt.show()

For every supported orientation, $\hat{\mathbf{r}}$, $\hat{\mathbf{c}}$, and $\hat{\mathbf{n}}$ are signed unit vectors along the patient axes, so the matrix is a signed permutation with scaling. Each patient coordinate therefore depends on exactly one stored index. The table below is computed from `ImageOrientationPatient` alone; with increasing slice offsets it matches the table in the validation note.

In [ ]:
def direction(vector):
    axis = int(np.argmax(np.abs(vector)))
    return f"{'+' if vector[axis] > 0 else '-'}{'xyz'[axis]}"


print(f"{'orientation':<12}{'columns run':<14}{'rows run':<11}{'slices run'}")
for name, iop in ORIENTATIONS.items():
    r, c = np.array(iop[:3]), np.array(iop[3:])
    print(f"{name:<12}{direction(r):<14}{direction(c):<11}{direction(np.cross(r, c))}")

### 1.1 How PyMedPhys applies the definition

Because the matrix is a signed permutation, PyMedPhys evaluates one axis per patient coordinate instead of a coordinate for every voxel: $x$, $y$, and $z$ each come from the single stored index they depend on, in $O(\text{rows} + \text{columns} + \text{slices})$ operations. `pymedphys.dicom.zyx_and_dose_from_dataset` then

1. transposes the stored (slice, row, column) dimensions into patient $(z, y, x)$ order, which swaps rows and columns for the decubitus orientations; and
2. reverses each axis that decreases, together with the matching dimension of the dose.

The result satisfies `dose[k, j, i]` $\leftrightarrow$ `(z[k], y[j], x[i])` with every axis ascending. Nothing is interpolated: each stored value is moved, unchanged, to the array position that matches its coordinates. Below, each stored voxel of a small grid is labelled with its storage index, so the rearrangement can be read directly.

In [ ]:
labels = np.arange(3 * 4 * 5).reshape(3, 4, 5)
fig, axs = plt.subplots(2, 2, figsize=(11, 7.4), layout="constrained")
for row, name in enumerate(["HFP", "HFDL"]):
    ds = make_rtdose(name, (40.0, -30.0, 12.0), labels, (2.0, 3.0), (0.0, 2.5, 5.0))
    (z, y, x), dose = pymedphys.dicom.zyx_and_dose_from_dataset(ds)
    returned = np.rint(dose[0] / DOSE_GRID_SCALING).astype(int)

    ax = axs[row, 0]
    ax.imshow(ds.pixel_array[0], cmap=DOSE_CMAP, vmin=-8, vmax=labels.max())
    for (i, j), value in np.ndenumerate(ds.pixel_array[0]):
        ax.text(j, i, value, ha="center", va="center", fontsize=9)
    ax.set_xticks(range(ds.Columns))
    ax.set_yticks(range(ds.Rows))
    ax.set_xlabel("stored column index j")
    ax.set_ylabel("stored row index i")
    ax.set_title(f"{name}: pixel_array[0], stored order {ds.pixel_array.shape}")

    ax = axs[row, 1]
    dx, dy = x[1] - x[0], y[1] - y[0]
    ax.imshow(
        returned, cmap=DOSE_CMAP, vmin=-8, vmax=labels.max(),
        extent=(x[0] - dx / 2, x[-1] + dx / 2, y[-1] + dy / 2, y[0] - dy / 2),
    )
    for (jy, ix), value in np.ndenumerate(returned):
        ax.text(x[ix], y[jy], value, ha="center", va="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_yticks(y)
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title(f"{name}: returned dose[0] at z = {z[0]:g} mm {dose.shape}")
fig.suptitle("Stored voxels (left) and the same voxels after zyx_and_dose_from_dataset (right)")
plt.show()

Voxel 0 is the first stored voxel, at $\mathbf{S} = (40, -30, 12)$ mm. For head first prone the columns run towards $-x$ and the rows towards $-y$, so both in-plane axes are reversed and voxel 0 moves to the corner with the largest $x$ and $y$. For head first decubitus left the rows run along $+x$ and the columns along $-y$, so the dimensions are also swapped and the returned array has shape (3, 5, 4) rather than (3, 4, 5).

## 2. The current conversion places every voxel correctly

### 2.1 Every voxel against the DICOM definition

The first check converts a grid with an off-centre origin, unequal row and column spacing, unequal numbers of rows and columns, and unevenly spaced slices, stored in each orientation with increasing and with decreasing slice offsets. It reports the largest distance between any returned voxel and the position that `voxel_positions` calculates for it.

In [ ]:
position, shape, pixel_spacing = (123.0, -187.0, 55.0), (3, 4, 5), (2.0, 3.0)
labels = np.arange(np.prod(shape)).reshape(shape)

print(f"{'orientation':<13}{'slice offsets':<15}{'current':<11}previous")
for name in ORIENTATIONS:
    for order, offsets in (("increasing", (0.0, 2.5, 6.0)), ("decreasing", (0.0, -2.5, -6.0))):
        ds = make_rtdose(name, position, labels, pixel_spacing, offsets)
        current = placement_error(ds, *pymedphys.dicom.zyx_and_dose_from_dataset(ds))
        previous = placement_error(ds, *legacy_zyx_and_dose_from_dataset(ds))
        previous_text = "axes did not fit the dose" if np.isnan(previous) else f"{previous:.1f} mm"
        print(f"{name:<13}{order:<15}{f'{current:g} mm':<11}{previous_text}")

The current conversion places every voxel exactly: the differences are zero, not merely small, because both calculations add the same signed multiples of the spacing to the same origin. The previous conversion was exact only for head first supine. For the other non-decubitus orientations it displaced the grid by hundreds of millimetres, and for the decubitus orientations its axes did not even have the lengths of the dose dimensions they were paired with.

### 2.2 Randomised grids

The same comparison over 800 random grids, 100 per orientation, with 2 to 8 voxels along each dimension, spacings from 0.2 mm to 5 mm, origins within ±1500 mm, and uneven slice offsets that increase or decrease. Some head first supine grids use absolute slice offsets.

In [ ]:
rng = np.random.default_rng(2066)
current_errors = {name: [] for name in ORIENTATIONS}
previous_errors = {name: [] for name in ORIENTATIONS}
for name in ORIENTATIONS:
    for _ in range(100):
        shape = tuple(int(n) for n in rng.integers(2, 9, size=3))
        spacing = np.round(rng.uniform(0.2, 5.0, size=2), 2)
        position = np.round(rng.uniform(-1500, 1500, size=3), 2)
        steps = np.round(rng.uniform(0.2, 5.0, size=shape[0] - 1), 2)
        offsets = np.concatenate([[0.0], np.cumsum(steps)])
        if rng.random() < 0.5:
            offsets = -offsets
        if name == "HFS" and rng.random() < 0.3:
            offsets = offsets + position[2]  # absolute z coordinates
        ds = make_rtdose(name, position, np.arange(np.prod(shape)).reshape(shape), spacing, offsets)
        current_errors[name].append(placement_error(ds, *pymedphys.dicom.zyx_and_dose_from_dataset(ds)))
        previous_errors[name].append(placement_error(ds, *legacy_zyx_and_dose_from_dataset(ds)))

worst_current = max(max(errors) for errors in current_errors.values())
print(f"Largest current placement error in 800 grids: {worst_current:g} mm")

fig, ax = plt.subplots(figsize=(11, 4.6), layout="constrained")
jitter = np.random.default_rng(0)
for index, name in enumerate(ORIENTATIONS):
    current = np.array(current_errors[name])
    ax.scatter(
        index - 0.2 + jitter.uniform(-0.12, 0.12, current.size), current,
        s=12, color=CURRENT, alpha=0.7, linewidths=0, label="current" if index == 0 else None,
    )
    errors = np.array(previous_errors[name])
    fitted = errors[np.isfinite(errors)]
    ax.scatter(
        index + 0.2 + jitter.uniform(-0.12, 0.12, fitted.size), fitted,
        s=12, color=PREVIOUS, alpha=0.7, linewidths=0, label="previous" if index == 0 else None,
    )
    unfit = int(np.sum(~np.isfinite(errors)))
    if unfit:
        ax.text(index + 0.2, 3, f"{unfit} of 100:\naxes did not\nfit the dose", ha="center", va="center", fontsize=8, color=SECONDARY)
ax.set_yscale("symlog", linthresh=1)
ax.set_ylim(-0.3, 1.5e4)
ax.set_xticks(range(len(ORIENTATIONS)), list(ORIENTATIONS))
ax.set_ylabel("largest voxel displacement (mm)")
ax.grid(axis="y")
ax.legend(loc="upper left", title="conversion")
ax.set_title("Largest voxel displacement in 100 random grids per orientation")
plt.show()

Head first supine grids with relative offsets were placed exactly before as well; the displaced head first supine grids are those with absolute slice offsets (section 3.4). Decubitus grids with equal numbers of rows and columns were accepted by the previous conversion but transposed in the transverse plane, so they appear as finite displacements.

### 2.3 One dose distribution, eight storage orders

A stronger test starts from a known physical dose distribution and stores it in each orientation. The storage directions below are tabulated by hand from section 1, and the encoding uses neither `voxel_positions` nor PyMedPhys. The distribution is shaped like the letter F, which has no mirror or rotational symmetry, so any flip or transposition would be visible. It also varies along z.

In [ ]:
def image_extent(x, y):
    """imshow extent for ascending axes, with y increasing downwards."""
    dx, dy = x[1] - x[0], y[1] - y[0]
    return (x[0] - dx / 2, x[-1] + dx / 2, y[-1] + dy / 2, y[0] - dy / 2)


def letter_f(y, x, shift_x=0.0):
    """Smooth dose (Gy) shaped like an F, read with y increasing downwards."""
    Y, X = np.meshgrid(y, x, indexing="ij")
    xc, yc, stroke = x.mean() + shift_x, y.mean(), 7.0
    left = xc - 12
    stem = (np.abs(X - left) <= stroke / 2) & (np.abs(Y - yc) <= 22)
    top = (np.abs(Y - (yc - 22 + stroke / 2)) <= stroke / 2) & (X >= left - stroke / 2) & (X <= xc + 14)
    middle = (np.abs(Y - yc) <= stroke / 2) & (X >= left - stroke / 2) & (X <= xc + 8)
    letter = (stem | top | middle).astype(float)
    smooth = scipy.ndimage.gaussian_filter(letter, sigma=(2.5 / (y[1] - y[0]), 2.5 / (x[1] - x[0])))
    return 0.1 + 1.9 * smooth / smooth.max()


# Stored (slice, row, column) dimensions as patient (z, y, x) dimensions, and
# the direction of travel along each, tabulated by hand from section 1.
STORAGE = {
    "HFS": ((0, 1, 2), (1, 1, 1)),
    "HFP": ((0, 1, 2), (1, -1, -1)),
    "FFS": ((0, 1, 2), (-1, 1, -1)),
    "FFP": ((0, 1, 2), (-1, -1, 1)),
    "HFDL": ((0, 2, 1), (1, 1, -1)),
    "HFDR": ((0, 2, 1), (1, -1, 1)),
    "FFDL": ((0, 2, 1), (-1, 1, 1)),
    "FFDR": ((0, 2, 1), (-1, -1, -1)),
}


def encode(orientation, axes_zyx, pixels_zyx, reverse_slices=False):
    """Store a physical grid, given with ascending (z, y, x) axes, in an orientation."""
    dimensions, signs = STORAGE[orientation]
    stored = np.transpose(pixels_zyx, dimensions)
    stored_axes = []
    for dimension, (patient_dimension, sign) in enumerate(zip(dimensions, signs)):
        axis = axes_zyx[patient_dimension]
        if sign < 0:
            stored = np.flip(stored, axis=dimension)
            axis = axis[::-1]
        stored_axes.append(axis)
    if reverse_slices:
        stored, stored_axes[0] = stored[::-1], stored_axes[0][::-1]
    position = np.empty(3)
    for patient_dimension, axis in zip(dimensions, stored_axes):
        position[2 - patient_dimension] = axis[0]
    return make_rtdose(
        orientation,
        position,
        stored,
        pixel_spacing=[abs(axis[1] - axis[0]) for axis in stored_axes[1:]],
        slice_offsets=(stored_axes[0] - stored_axes[0][0]) * signs[0],
    )


# The hand-written table must agree with ImageOrientationPatient.
for name, (dimensions, signs) in STORAGE.items():
    r, c = np.array(ORIENTATIONS[name][:3]), np.array(ORIENTATIONS[name][3:])
    for vector, dimension, sign in zip((np.cross(r, c), c, r), dimensions, signs):
        axis = int(np.argmax(np.abs(vector)))
        assert dimension == 2 - axis and sign == np.sign(vector[axis]), name
print("The hand-tabulated storage directions agree with ImageOrientationPatient.")

In [ ]:
axes_true = (15.0 + 3.0 * np.arange(5), -56.0 + 2.0 * np.arange(32), -8.75 + 2.5 * np.arange(32))
z_true, y_true, x_true = axes_true
profile_z = np.exp(-0.5 * ((z_true - z_true.mean()) / 6.0) ** 2)
pixels_true = np.rint(letter_f(y_true, x_true)[None] * profile_z[:, None, None] / DOSE_GRID_SCALING).astype(np.uint32)
dose_true = pixels_true * DOSE_GRID_SCALING
middle = 2  # the central slice, z = 21 mm, in every storage order

encoded = {name: encode(name, axes_true, pixels_true) for name in ORIENTATIONS}
recovered = []
for name in ORIENTATIONS:
    for reverse_slices in (False, True):
        axes, dose = pymedphys.dicom.zyx_and_dose_from_dataset(encode(name, axes_true, pixels_true, reverse_slices))
        recovered.append(all(np.array_equal(a, b) for a, b in zip(axes, axes_true)) and np.array_equal(dose, dose_true))
print(f"{sum(recovered)} of {len(recovered)} encodings recover exactly the same axes and dose")

fig, axs = plt.subplots(2, 8, figsize=(15, 4.1), layout="constrained")
for column, (name, ds) in enumerate(encoded.items()):
    ax = axs[0, column]
    ax.imshow(ds.pixel_array[middle] * DOSE_GRID_SCALING, cmap=DOSE_CMAP, vmin=0, vmax=2)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
    (z, y, x), dose = pymedphys.dicom.zyx_and_dose_from_dataset(ds)
    ax = axs[1, column]
    ax.imshow(dose[middle], cmap=DOSE_CMAP, vmin=0, vmax=2, extent=image_extent(x, y))
    ax.set_xticks([0, 50])
    ax.set_yticks([-50, 0])
    ax.set_xlabel("x (mm)")
axs[0, 0].set_ylabel("stored pixel_array\n(row, column)")
axs[1, 0].set_ylabel("returned dose\ny (mm)")
fig.suptitle("The same dose stored in eight orientations (top) and after zyx_and_dose_from_dataset (bottom), central slice")
plt.show()

The stored images differ by rotations, reflections, and transpositions; the returned arrays are identical to each other and to the original, bit for bit, in every orientation and for both slice orders.

### 2.4 Where the previous conversion placed the same dose

The figure overlays the 50% isodose line of the central slice as placed by each conversion, on top of the true position.

In [ ]:
from matplotlib.lines import Line2D

level = [0.5 * dose_true.max()]
fig, axs = plt.subplots(2, 4, figsize=(12.5, 6.3), sharex=True, sharey=True, layout="constrained")
for ax, (name, ds) in zip(axs.flat, encoded.items()):
    (z, y, x), dose = pymedphys.dicom.zyx_and_dose_from_dataset(ds)
    ax.contour(x, y, dose[middle], levels=level, colors=CURRENT, linewidths=3, zorder=2)
    (z_old, y_old, x_old), stored = legacy_zyx_and_dose_from_dataset(ds)
    # The previous pairing: stored rows with y_old and stored columns with x_old.
    ax.contour(x_old, y_old, stored[middle], levels=level, colors=PREVIOUS, linewidths=2, zorder=3)
    ax.contour(x_true, y_true, dose_true[middle], levels=level, colors=TRUTH, linestyles="dashed", linewidths=1.2, zorder=4)
    if name == "HFS":
        ax.text(0, 30, "all three coincide", ha="center", color=SECONDARY)
    ax.set_title(f"{name}\nprevious centre ({x_old.mean():+.0f}, {y_old.mean():+.0f}, {z_old.mean():+.0f}) mm")
    ax.set_aspect("equal")
    ax.grid(True)
axs[0, 0].set_xlim(-75, 75)
axs[0, 0].set_ylim(60, -60)
for ax in axs[1]:
    ax.set_xlabel("x (mm)")
for ax in axs[:, 0]:
    ax.set_ylabel("y (mm)")
fig.legend(
    handles=[
        Line2D([], [], color=TRUTH, linestyle="dashed", linewidth=1.2, label="DICOM position"),
        Line2D([], [], color=CURRENT, label="current conversion"),
        Line2D([], [], color=PREVIOUS, label="previous conversion"),
    ],
    loc="outside lower center", ncols=3,
)
fig.suptitle(f"50% isodose line of the central slice; true centre ({x_true.mean():+.0f}, {y_true.mean():+.0f}, {z_true.mean():+.0f}) mm")
plt.show()

The current conversion (blue) coincides with the DICOM position (dashed) in every orientation. The previous conversion (orange):

- placed head first supine correctly;
- for head first prone, feet first supine, and feet first prone, reflected the grid centre through the origin along each reversed axis, so the dose was **translated**, not mirrored, by twice the centre coordinate: $x' = x - 2c_x$ on a reversed x axis, with $c_x$ the centre of the grid;
- for the decubitus orientations, also **transposed** the transverse plane, because it paired the stored rows with $y$ and the columns with $x$.

Why the translation? For a reversed axis the true coordinates in storage order are $S - j\Delta$. The previous code formed the image-aligned axis $-S + j\Delta$, appropriate for its IEC fixed output, and then reversed the array for DICOM output. Reversing restores the direction of travel but not the sign of the origin, so storage index $j$ received $-S + (n - 1 - j)\Delta$, which differs from the truth by $(n - 1)\Delta - 2S = -2c$.

## 3. Edge cases

### 3.1 Cropping and shifting a grid

Because the error is $-2c$, it depends on where the grid is and how far it extends. The smallest example is a single head first prone row of five 1 mm columns starting at $x = 100$ mm, with and without its last two columns.

In [ ]:
def hfp_row(columns):
    return make_rtdose("HFP", (100.0, 0.0, 0.0), np.arange(columns)[None, None, :], (1.0, 1.0), [0.0])


for label, ds in (("five columns", hfp_row(5)), ("three columns", hfp_row(3))):
    true_x = voxel_positions(ds)[0, 0, :, 0]
    previous_x = legacy_xyz_axes_from_dataset(ds)[0]
    (_, _, current_x), dose = pymedphys.dicom.zyx_and_dose_from_dataset(ds)
    stored = np.rint(dose[0, 0] / DOSE_GRID_SCALING).astype(int)
    print(f"{label}:")
    print(f"  DICOM x of stored columns 0, 1, ...    {true_x}")
    print(f"  previous x of stored columns 0, 1, ... {previous_x}   error {previous_x[0] - true_x[0]:+g} mm")
    print(f"  current ascending x {current_x}, holding stored columns {stored}")

The previous error is $-196$ mm for the full row and $-198$ mm for the cropped one, so the two grids, **both head first prone**, disagree by 2 mm. Grids that share the same geometry carry the same error, which cancels when they are compared with each other; crops, grids with different extents such as a treatment planning grid and a Monte Carlo grid, and shifted grids do not.

The next figure repeats this on a three-dimensional head first prone grid with a smoothed 50 mm × 40 mm field: once with 10 mm removed from its $+x$ edge, and once with the whole grid, and its dose, moved 15 mm towards $+x$.

In [ ]:
def box_field(z, y, x, centre_x=25.0):
    """A smoothed 50 mm x 40 mm field with a gentle wedge, in Gy."""
    Z, Y, X = np.meshgrid(z, y, x, indexing="ij")
    inside = (np.abs(X - centre_x) <= 25) & (np.abs(Y) <= 20)
    spacing = [axis[1] - axis[0] for axis in (z, y, x)]
    smooth = scipy.ndimage.gaussian_filter(inside.astype(float), [3.0 / s for s in spacing])
    return 0.05 + 1.9 * smooth * (1 + 0.004 * (X - centre_x)) * np.exp(-0.5 * (Z / 15.0) ** 2)


def profile_at(axes_zyx, dose_zyx, y0=0.0, z0=0.0):
    """x axis and dose along the stored row and slice labelled nearest to (y0, z0)."""
    z, y, x = axes_zyx
    return x, dose_zyx[np.argmin(np.abs(z - z0)), np.argmin(np.abs(y - y0)), :]


def true_profile(ds, y0=0.0, z0=0.0):
    """DICOM x positions and dose of the stored voxels nearest to (y0, z0)."""
    positions = voxel_positions(ds)
    k = np.argmin(np.abs(positions[:, 0, 0, 2] - z0))
    i = np.argmin(np.abs(positions[k, :, 0, 1] - y0))
    return positions[k, i, :, 0], ds.pixel_array[k, i, :] * DOSE_GRID_SCALING


def x_at_half_maximum(x, profile):
    """Position of the rising 50% edge, whichever way x runs."""
    order = np.argsort(x)
    x, profile = x[order], profile[order]
    rising = np.argmax(profile >= 0.5 * profile.max())
    return np.interp(0.5 * profile.max(), profile[rising - 1 : rising + 1], x[rising - 1 : rising + 1])


box_axes = (-10.0 + 2.5 * np.arange(9), -40.0 + 2.5 * np.arange(33), -30.0 + 2.5 * np.arange(41))
box_pixels = np.rint(box_field(*box_axes) / DOSE_GRID_SCALING).astype(np.uint32)
bz, by, bx = box_axes
full = encode("HFP", box_axes, box_pixels)
cropped = encode("HFP", (bz, by, bx[:-4]), box_pixels[..., :-4])
shifted = encode("HFP", (bz, by, bx + 15.0), box_pixels)

conversions = {
    "previous": (legacy_zyx_and_dose_from_dataset, PREVIOUS),
    "current": (pymedphys.dicom.zyx_and_dose_from_dataset, CURRENT),
}
cases = {
    "crop": (cropped, "10 mm removed from the +x edge"),
    "shift": (shifted, "grid moved 15 mm towards +x"),
}
fig, axs = plt.subplots(2, 2, figsize=(12, 7.4), sharey=True, layout="constrained")
for row, (case, (changed, description)) in enumerate(cases.items()):
    for column, (label, (convert, colour)) in enumerate(conversions.items()):
        ax = axs[row, column]
        edge = {}
        for ds, style, name in ((full, "-", "original grid"), (changed, "--", description)):
            x, profile = profile_at(*convert(ds))
            ax.plot(x, profile, style, color=colour, label=name)
            ax.plot(*true_profile(ds), ":", color=TRUTH, linewidth=1)
            edge[name] = x_at_half_maximum(x, profile)
        moved = edge[description] - edge["original grid"]
        outcome = "did not move" if abs(moved) < 0.5 else f"moved {moved:+.0f} mm"
        ax.set_title(f"{label} conversion, {case}: the dose {outcome}")
        ax.legend(loc="upper left", ncols=2)
        ax.set_ylim(0, 2.6)
        ax.grid(True)
        ax.set_xlim(-110, 110)
for ax in axs[1]:
    ax.set_xlabel("x (mm)")
for ax in axs[:, 0]:
    ax.set_ylabel("dose (Gy) at y = 0, z = 0")
fig.suptitle("Head first prone profiles; dotted black lines show the DICOM positions")
plt.show()

The current conversion reports the crop as no movement and the shift as $+15$ mm, as it should. The previous conversion moved the cropped dose by 10 mm and moved the shifted dose by $-15$ mm, in the wrong direction; both previous profiles also sit far from their true positions (dotted).

Gamma inherits these errors when its coordinates come from the conversion. Below, the cropped grid is the reference and the full grid the evaluation, at 3%/3 mm with a 10% lower dose cutoff. The previous coordinates are passed to the current `pymedphys.gamma`, which accepts their descending axes; with its default interpolator, version 0.41.0 raised an error for this evaluation grid instead (section 4.3), and with `interp_algo="scipy"` it returned the same values as below.

In [ ]:
gamma_options = {
    "dose_percent_threshold": 3,
    "distance_mm_threshold": 3,
    "lower_percent_dose_cutoff": 10,
    "max_gamma": 2,
}
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained")
for ax, (label, (convert, colour)) in zip(axs, conversions.items()):
    axes_reference, dose_reference = convert(cropped)
    axes_evaluation, dose_evaluation = convert(full)
    gamma = pymedphys.gamma(axes_reference, dose_reference, axes_evaluation, dose_evaluation, **gamma_options)
    valid = gamma[~np.isnan(gamma)]
    z, y, x = axes_reference
    k = np.argmin(np.abs(z))
    mesh = ax.pcolormesh(x, y, gamma[k], shading="nearest", cmap=GAMMA_CMAP, vmin=0, vmax=2)
    ax.contour(x, y, dose_reference[k], levels=[0.5 * dose_reference.max()], colors=TRUTH, linewidths=1)
    ax.set_aspect("equal")
    ax.set_ylim(max(y) + 1.25, min(y) - 1.25)
    ax.set_xlabel("x (mm), as reported by the conversion")
    ax.set_ylabel("y (mm)")
    ax.set_title(f"{label} conversion: {100 * np.mean(valid <= 1):.1f}% of points pass")
fig.colorbar(mesh, ax=axs, label="γ (3%/3 mm)")
fig.suptitle("Gamma of the cropped grid against the full grid, z = 0; black line: 50% isodose of the reference")
plt.show()

With the current conversion, gamma of a grid against its own crop is zero everywhere. With the previous one, the artificial 10 mm shift fails the high-gradient edges.

### 3.2 Crops of every axis in every orientation

The same comparison, summarised: for each orientation, 10 mm is removed from either end of each patient axis, and the table reports the largest change in where the conversion places the voxels that the two grids share.

In [ ]:
def legacy_displacement(ds):
    """Displacement (mm) that the previous conversion gave each stored voxel.

    Returns an array indexed [slice, row, column, xyz]. Uses only the geometry
    attributes, so it works without decoding the pixel data. Raises ValueError
    where the previous axes did not fit the stored dose.
    """
    x, y, z = legacy_xyz_axes_from_dataset(ds)
    stored_shape = (int(getattr(ds, "NumberOfFrames", 1)), int(ds.Rows), int(ds.Columns))
    if (len(z), len(y), len(x)) != stored_shape:
        raise ValueError(f"previous axes {(len(z), len(y), len(x))} do not fit the stored dose {stored_shape}")
    Z, Y, X = np.meshgrid(z, y, x, indexing="ij")
    return np.stack([X, Y, Z], axis=-1) - voxel_positions(ds)


def current_displacement(ds):
    """Displacement (mm) that the current conversion gives each stored voxel."""
    labelled = copy.deepcopy(ds)  # label every stored voxel with its storage index
    labelled.PixelData = np.arange(ds.pixel_array.size, dtype="<u4").tobytes()
    true = voxel_positions(ds)
    returned = returned_positions(labelled, *pymedphys.dicom.zyx_and_dose_from_dataset(labelled))
    return returned.reshape(true.shape) - true


def largest_relative_shift(ds_a, ds_b, displacement):
    """Largest change in placement of the voxels that two grids share."""
    placed = []
    for ds in (ds_a, ds_b):
        true = voxel_positions(ds).reshape(-1, 3)
        moved = displacement(ds).reshape(-1, 3)
        placed.append({tuple(np.round(t, 6)): m for t, m in zip(true, moved)})
    shared = placed[0].keys() & placed[1].keys()
    return max(np.linalg.norm(placed[0][key] - placed[1][key]) for key in shared)


crop_axes = (100.0 + 2.5 * np.arange(12), -70.0 + 2.5 * np.arange(24), 10.0 + 2.5 * np.arange(24))
crop_labels = np.zeros((12, 24, 24), dtype=np.uint32)  # relabelled where needed
crops = [(dimension, end) for dimension in (2, 1, 0) for end in ("low", "high")]
previous_shift = np.full((len(ORIENTATIONS), len(crops)), np.nan)
current_shift = np.zeros_like(previous_shift)
for row, name in enumerate(ORIENTATIONS):
    full_grid = encode(name, crop_axes, crop_labels)
    for column, (dimension, end) in enumerate(crops):
        keep = [slice(None)] * 3
        keep[dimension] = slice(4, None) if end == "low" else slice(None, -4)
        axes = tuple(axis[keep[d]] for d, axis in enumerate(crop_axes))
        crop_grid = encode(name, axes, crop_labels[tuple(keep)])
        current_shift[row, column] = largest_relative_shift(full_grid, crop_grid, current_displacement)
        try:
            previous_shift[row, column] = largest_relative_shift(full_grid, crop_grid, legacy_displacement)
        except ValueError:
            pass
print(f"Largest current shift over all {current_shift.size} crops: {current_shift.max():g} mm")

fig, ax = plt.subplots(figsize=(9.2, 5.4), layout="constrained")
cmap = DOSE_CMAP.with_extremes(bad="#e1e0d9")
image = ax.imshow(np.ma.masked_invalid(previous_shift), cmap=cmap, vmin=0, vmax=12)
for (row, column), value in np.ndenumerate(previous_shift):
    text = "axes did\nnot fit" if np.isnan(value) else f"{value:.0f} mm"
    ax.text(column, row, text, ha="center", va="center", fontsize=8, color="white" if value > 6 else TRUTH)
ax.set_xticks(range(len(crops)), [f"{'zyx'[d]}, {end}" for d, end in crops])
ax.set_xlabel("patient axis and end from which 10 mm was removed")
ax.set_yticks(range(len(ORIENTATIONS)), list(ORIENTATIONS))
fig.colorbar(image, ax=ax, label="previous shift (mm)")
ax.set_title("Previous conversion: shift between a grid and its 10 mm crop\n(current conversion: 0 mm in all 48 cases)")
plt.show()

A crop moved the previous placement by the width removed whenever it was taken along a reversed axis, and not otherwise. For decubitus grids, an in-plane crop made the numbers of rows and columns unequal, and the previous axes no longer fitted the dose at all.

### 3.3 Decreasing slice offsets

A head first supine grid may list its slices in decreasing z. The previous conversion placed these slices correctly but returned a descending z axis, which the default gamma interpolator of 0.41.0 could not use as an evaluation grid (section 4.3). The current conversion returns an ascending axis with the slices reversed to match.

In [ ]:
slice_index = np.arange(4)[:, None, None] * np.ones((1, 2, 2), dtype=int)
ds = make_rtdose("HFS", (0.0, 0.0, 30.0), slice_index, (1.0, 1.0), [0.0, -3.0, -6.0, -9.0])
(z, _, _), dose = pymedphys.dicom.zyx_and_dose_from_dataset(ds)
print("DICOM z of stored slices 0, 1, 2, 3:", voxel_positions(ds)[:, 0, 0, 2])
print("previous z axis:                  ", legacy_xyz_axes_from_dataset(ds)[2])
print("current z axis:                   ", z)
print("stored slice at each current z:   ", np.rint(dose[:, 0, 0] / DOSE_GRID_SCALING).astype(int))

### 3.4 Absolute slice offsets

When the first element of `GridFrameOffsetVector` is not zero, the vector holds absolute z coordinates, which is permitted only for `ImageOrientationPatient` $= (1, 0, 0, 0, 1, 0)$. The previous conversion added them to $S_z$ as if they were offsets, displacing every slice by $S_z$.

In [ ]:
four_slices = np.arange(4 * 2 * 2).reshape(4, 2, 2)
relative = make_rtdose("HFS", (0.0, 0.0, -250.0), four_slices, (1.0, 1.0), [0.0, 3.0, 6.0, 9.0])
absolute = make_rtdose("HFS", (0.0, 0.0, -250.0), four_slices, (1.0, 1.0), [-250.0, -247.0, -244.0, -241.0])
for label, ds in (("relative offsets", relative), ("absolute offsets", absolute)):
    print(f"{label}: DICOM z {voxel_positions(ds)[:, 0, 0, 2]}")
    print(f"  previous z {legacy_xyz_axes_from_dataset(ds)[2]}")
    print(f"  current z  {pymedphys.dicom.zyx_and_dose_from_dataset(ds)[0][0]}")

prone = make_rtdose("HFP", (0.0, 0.0, -250.0), four_slices, (1.0, 1.0), [-250.0, -247.0, -244.0, -241.0])
try:
    pymedphys.dicom.zyx_and_dose_from_dataset(prone)
except ValueError as error:
    print(f"\nAbsolute offsets with another orientation are rejected:\n  {error}")

### 3.5 Single-slice RT Dose

A planar RT Dose, such as an export for a detector-array measurement, has one slice. pydicom returns its pixel data as a two-dimensional array, and the standard allows `GridFrameOffsetVector` to be absent. The previous conversion raised an error without the vector; with a single-valued vector, it returned z as a scalar alongside a two-dimensional dose, which gamma and `dicom_dose_interpolate` then rejected. The current conversion returns a three-dimensional array with a length-one z axis at $S_z$.

The gamma shell search is unreliable when an evaluation axis has a single point ([#2070](https://github.com/pymedphys/pymedphys/issues/2070)), so compare planes with two-dimensional axes and arrays, as below.

In [ ]:
plane_y, plane_x = -20.0 + 2.0 * np.arange(21), -30.0 + 2.5 * np.arange(25)
plane = np.rint(letter_f(plane_y, plane_x) / DOSE_GRID_SCALING).astype(np.uint32)[None]
planar = make_rtdose("HFS", (-30.0, -20.0, 42.0), plane, (2.0, 2.5))  # no GridFrameOffsetVector
print("pixel_array shape:", planar.pixel_array.shape)
try:
    legacy_zyx_and_dose_from_dataset(planar)
except AttributeError as error:
    print("previous conversion:", type(error).__name__, "-", error)
(z, y, x), dose = pymedphys.dicom.zyx_and_dose_from_dataset(planar)
print("current conversion: dose shape", dose.shape, "with z axis", z)

measured = np.roll(dose[0], 1, axis=1) * 1.01  # a 2.5 mm shift and a 1% scaling
gamma_2d = pymedphys.gamma((y, x), dose[0], (y, x), measured, 3, 3, lower_percent_dose_cutoff=10)
valid = gamma_2d[~np.isnan(gamma_2d)]
print(f"two-dimensional gamma, 3%/3 mm: {100 * np.mean(valid <= 1):.1f}% of points pass")

### 3.6 Oblique orientations

Only the eight axis-aligned orientations can be described by three independent patient axes. Direction cosines within $10^{-4}$ of an axis-aligned orientation are accepted, which allows for rounding in the file; oblique grids are rejected rather than misplaced.

In [ ]:
angle = np.radians(5)
tilted = make_rtdose("HFS", (0.0, 0.0, 0.0), np.zeros((2, 3, 3)), (1.0, 1.0), [0.0, 1.0])
tilted.ImageOrientationPatient = [round(np.cos(angle), 6), round(np.sin(angle), 6), 0, round(-np.sin(angle), 6), round(np.cos(angle), 6), 0]
try:
    pymedphys.dicom.zyx_and_dose_from_dataset(tilted)
except ValueError as error:
    print(error)

### 3.7 Summing doses needs the same pixel mapping

Summing RT Dose files, as the experimental Sum Coincident DICOM Doses app does, adds the stored pixel arrays element by element, which is valid only if equal indices mean equal positions. The previous check compared the axis values and the array shape. Below, a head first supine grid and a feet first decubitus left grid describe the same positions, centred on $z = 0$, but the decubitus grid stores $x$ along its rows. Their axis values agree, yet adding the arrays adds each dose to its own transpose.

In [ ]:
sum_axes = (-5.0 + 2.5 * np.arange(5), -30.0 + 2.5 * np.arange(24), -28.75 + 2.5 * np.arange(24))
sum_profile = np.exp(-0.5 * (sum_axes[0] / 6.0) ** 2)[:, None, None]
sum_pixels = np.rint(letter_f(sum_axes[1], sum_axes[2])[None] * sum_profile / DOSE_GRID_SCALING).astype(np.uint32)
supine = encode("HFS", sum_axes, sum_pixels)
decubitus = encode("FFDL", sum_axes, sum_pixels, reverse_slices=True)
for ds in (supine, decubitus):
    ds.PatientID = "SYNTHETIC"

print("previous check, coordinates equal:", legacy_coords_in_datasets_are_equal([supine, decubitus]))
print("current check, coordinates equal: ", coords_in_datasets_are_equal([supine, decubitus]))

stored_sum = (supine.pixel_array + decubitus.pixel_array) * DOSE_GRID_SCALING
_, supine_dose = pymedphys.dicom.zyx_and_dose_from_dataset(supine)
_, decubitus_dose = pymedphys.dicom.zyx_and_dose_from_dataset(decubitus)
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2), layout="constrained")
extent = image_extent(sum_axes[2], sum_axes[1])
for ax, image, title in (
    (axs[0], stored_sum[2], "stored arrays added, as the previous check allowed"),
    (axs[1], (supine_dose + decubitus_dose)[2], "doses added on the patient axes"),
):
    shown = ax.imshow(image, cmap=DOSE_CMAP, vmin=0, vmax=4, extent=extent)
    ax.set_title(title)
    ax.set_xlabel("x (mm)")
axs[0].set_ylabel("y (mm)")
fig.colorbar(shown, ax=axs, label="dose (Gy)")
fig.suptitle("Two copies of the same dose, z = 0")
plt.show()

The current check compares the complete mapping from pixel indices to patient positions: the direction cosines as well as the axes, with corresponding voxel centres required to agree within 0.01 mm in three dimensions. The previous check also used `numpy.allclose` with its default relative tolerance, so the permitted mismatch grew with distance from the coordinate origin.

## 4. How gamma treats axis order

`pymedphys.gamma(axes_reference, dose_reference, axes_evaluation, dose_evaluation, ...)` uses its two grids differently.

- The **reference** grid supplies the points at which gamma is evaluated. It is used as given, in either order, and gamma is returned with the shape and index order of `dose_reference`: `gamma[k, j, i]` belongs to the same position as `dose_reference[k, j, i]`.
- The **evaluation** grid is interpolated on shells of increasing radius around each reference point. The default interpolator needs strictly ascending, evenly spaced axes with at least two points each, so gamma reverses each descending evaluation axis together with the matching dimension of the evaluation dose before the search starts. Unevenly spaced evaluation axes are passed to the SciPy interpolator instead, with a warning.

### 4.1 Reversing an axis keeps each coordinate with its dose

The preparation step is `_prepare_evaluation_grid`. For a descending axis it reverses the coordinates and the dose together, so every dose value stays at its coordinate.

In [ ]:
stored_x = np.array([100.0, 99.0, 98.0])
stored_dose = np.array([10.0, 20.0, 30.0])
(prepared_x,), prepared_dose, algorithm = _prepare_evaluation_grid((stored_x,), stored_dose, "pymedphys")
print("as supplied:", stored_x, stored_dose)
print("prepared:   ", prepared_x, prepared_dose, f"(interpolator: {algorithm})")

fig, ax = plt.subplots(figsize=(8, 2.9), layout="constrained")
for level, (xs, doses, label) in enumerate(((stored_x, stored_dose, "as supplied"), (prepared_x, prepared_dose, "prepared"))):
    height = 1 - level
    for index, (x_value, dose_value) in enumerate(zip(xs, doses)):
        ax.add_patch(plt.Rectangle((index - 0.45, height - 0.28), 0.9, 0.56, facecolor="#cde2fb", edgecolor="#fcfcfb", linewidth=2))
        ax.text(index, height, f"x = {x_value:g} mm\n{dose_value:g} Gy", ha="center", va="center")
    ax.text(-0.6, height, f"{label}\nindex 0, 1, 2", ha="right", va="center", color=SECONDARY)
for top_index, x_value in enumerate(stored_x):
    bottom_index = int(np.flatnonzero(prepared_x == x_value)[0])
    ax.annotate("", xy=(bottom_index, 0.3), xytext=(top_index, 0.7), arrowprops={"arrowstyle": "-|>", "color": MUTED})
ax.set_xlim(-2.2, 2.6)
ax.set_ylim(-0.4, 1.4)
ax.axis("off")
ax.set_title("Reversing a descending evaluation axis moves each (x, dose) pair intact")
plt.show()

### 4.2 Storage order does not change gamma

Below, the same evaluation dose is supplied in four storage orders. The four gamma results are identical, element for element. A reference supplied with a descending axis gives the same values in its own order, so it must be plotted against its own axes.

In [ ]:
g_y, g_x = -24.0 + 1.0 * np.arange(49), -28.0 + 1.0 * np.arange(57)
reference_2d = letter_f(g_y, g_x)
evaluation_2d = 1.02 * letter_f(g_y, g_x, shift_x=2.5)
options_2d = {"dose_percent_threshold": 3, "distance_mm_threshold": 2, "lower_percent_dose_cutoff": 10}

orders = {
    "evaluation, ascending": (slice(None), slice(None)),
    "evaluation, x descending": (slice(None), slice(None, None, -1)),
    "evaluation, y descending": (slice(None, None, -1), slice(None)),
    "evaluation, both descending": (slice(None, None, -1), slice(None, None, -1)),
}
gammas = {
    label: pymedphys.gamma((g_y, g_x), reference_2d, (g_y[rows], g_x[columns]), evaluation_2d[rows, columns], **options_2d)
    for label, (rows, columns) in orders.items()
}
first = gammas["evaluation, ascending"]
print("all four evaluation orders give identical gamma:", all(np.array_equal(g, first, equal_nan=True) for g in gammas.values()))

reversed_reference = pymedphys.gamma((g_y, g_x[::-1]), reference_2d[:, ::-1], (g_y, g_x), evaluation_2d, **options_2d)
print("reference with x descending gives the same values in its own order:", np.array_equal(reversed_reference[:, ::-1], first, equal_nan=True))

panels = [(label, evaluation_2d[rows, columns], gammas[label], g_x) for label, (rows, columns) in orders.items()]
panels.append(("reference, x descending", reference_2d[:, ::-1], reversed_reference, g_x[::-1]))
fig, axs = plt.subplots(2, 5, figsize=(15, 5.4), layout="constrained")
for column, (label, supplied, gamma, x_axis) in enumerate(panels):
    ax = axs[0, column]
    ax.imshow(supplied, cmap=DOSE_CMAP, vmin=0, vmax=2.1)
    ax.set_title(label)
    ax.set_xticks([])
    ax.set_yticks([])
    ax = axs[1, column]
    mesh = ax.pcolormesh(x_axis, g_y, gamma, shading="nearest", cmap=GAMMA_CMAP, vmin=0, vmax=2)
    ax.set_aspect("equal")
    ax.set_ylim(g_y.max() + 0.5, g_y.min() - 0.5)
    ax.set_xlabel("x (mm)")
axs[0, 0].set_ylabel("array as supplied\n(index order)")
axs[1, 0].set_ylabel("gamma on the reference axes\ny (mm)")
fig.colorbar(mesh, ax=axs[1], label="γ (3%/2 mm)", shrink=0.9)
fig.suptitle("Gamma of an F shifted by 2.5 mm: supplied arrays (top) and gamma plotted on the reference coordinates (bottom)")
plt.show()

### 4.3 Why descending evaluation axes failed before

The interpolation kernels locate a point by testing `axis[0] <= point <= axis[-1]` and searching the axis in ascending order. On a descending axis no point passes that test, so every interpolated value is the fill value, which gamma sets to infinity:

In [ ]:
descending = np.array([100.0, 99.0, 98.0])
values = np.array([10.0, 20.0, 30.0])
points = np.array([[98.5], [99.0], [99.5]])
print("descending axis:", interp_linear_1d(descending, values, points, np.inf))
print("ascending axis: ", interp_linear_1d(descending[::-1].copy(), values[::-1].copy(), points, np.inf))

A search that never finds a finite value returned NaN when `max_gamma` was set, and otherwise never terminated. In version 0.41.0 most affected DICOM grids did not get that far. The previous conversion produced its reversed axes with `numpy.flip`, which returns a view with negative strides, and passed them to the Numba kernel alongside contiguous axes. Numba cannot iterate over a tuple of arrays with different memory layouts, so the call failed:

In [ ]:
z_axis = np.array([0.0, 2.5, 5.0])  # contiguous, as for head first prone
y_axis = np.flip(np.array([-6.0, -4.0, -2.0, 0.0]))  # reversed views, as the
x_axis = np.flip(np.array([10.0, 12.0, 14.0, 16.0, 18.0]))  # previous conversion returned
try:
    interp_linear_3d((z_axis, y_axis, x_axis), np.zeros((3, 4, 5)), np.array([[2.0, -3.0, 13.0]]), np.inf)
except numba.core.errors.TypingError as error:
    message = [line.strip() for line in str(error).splitlines() if line.strip()]
    print(f"{type(error).__name__}: {message[0]}\n  {message[1]}")

When all three axes were reversed, as for feet first decubitus right, the layouts matched and the search ran without finding a value. The current gamma converts every evaluation axis to a contiguous ascending array before the search, so neither failure can occur.

### 4.4 The search is bounded by the two grids

The search starts at the smallest distance between the reference and evaluation grids and stops beyond the largest, or at `max_gamma` times the largest distance threshold if that is smaller. Grids that do not overlap still have valid gamma values: below, every reference point lies at least 4 mm from the evaluation grid, so the search starts at 4 mm, and every gamma value is at least $4/3$ at 3 mm.

In [ ]:
reference_x = np.arange(0.0, 10.1, 1.0)
evaluation_x = np.arange(14.0, 30.1, 1.0)
nearest, farthest = _grid_distance_bounds((reference_x,), (evaluation_x,))
print(f"closest possible distance {nearest:g} mm, farthest {farthest:g} mm")
reference_1d, evaluation_1d = 1.0 + 0.01 * reference_x, 1.0 + 0.01 * evaluation_x
gamma_1d = pymedphys.gamma((reference_x,), reference_1d, (evaluation_x,), evaluation_1d, 3, 3, lower_percent_dose_cutoff=0)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, layout="constrained")
top.plot(reference_x, reference_1d, "o-", color=CURRENT, markersize=5, label="reference grid")
top.plot(evaluation_x, evaluation_1d, "s-", color=MUTED, markersize=5, label="evaluation grid")
top.set_ylabel("dose (Gy)")
top.legend(loc="upper left")
bottom.plot(reference_x, gamma_1d, "o-", color=CURRENT, markersize=5, label="γ at each reference point")
bottom.plot(reference_x, (evaluation_x[0] - reference_x) / 3, ":", color=TRUTH, linewidth=1.2, label="distance to the evaluation grid / 3 mm")
bottom.set_xlabel("x (mm)")
bottom.set_ylabel("γ (3%/3 mm)")
bottom.legend(loc="upper right")
for ax in (top, bottom):
    ax.grid(True)
fig.suptitle("Gamma between grids that do not overlap")
plt.show()

## 5. Plotting gamma after the reordering

Gamma is returned on the reference grid, so plot it against the reference axes: `gamma[k, j, i]` is at `(z_ref[k], y_ref[j], x_ref[i])`, exactly like `dose_reference[k, j, i]`. The example below uses a head first prone reference and a head first decubitus right evaluation of the same F, shifted by 3 mm and scaled by 2%.

To overlay gamma on the stored pixel data instead, rearrange it into stored order first. `to_storage_order` applies the inverse of the conversion's transposition and reversals, using only `ImageOrientationPatient` and the direction of the slice offsets.

In [ ]:
def to_storage_order(array_zyx, ds):
    """Rearrange an array on the zyx_and_dose_from_dataset grid into stored order."""
    iop = np.rint(np.array(ds.ImageOrientationPatient, dtype=float))
    r, c = iop[:3], iop[3:]
    offsets = getattr(ds, "GridFrameOffsetVector", None)
    offsets = np.atleast_1d(np.array(offsets if offsets else [0.0], dtype=float))
    slice_sign = -1 if offsets.size > 1 and offsets[1] < offsets[0] else 1
    # Direction of travel along the stored slices, rows, and columns.
    directions = (slice_sign * np.cross(r, c), c, r)
    stored = np.transpose(array_zyx, [2 - int(np.argmax(np.abs(d))) for d in directions])
    for dimension, d in enumerate(directions):
        if d[np.argmax(np.abs(d))] < 0:
            stored = np.flip(stored, axis=dimension)
    return stored


round_trips = [
    np.array_equal(to_storage_order(pymedphys.dicom.zyx_and_dose_from_dataset(ds)[1], ds), ds.pixel_array * DOSE_GRID_SCALING)
    for name in ORIENTATIONS
    for ds in (encode(name, axes_true, pixels_true), encode(name, axes_true, pixels_true, reverse_slices=True))
]
print(f"to_storage_order restores pixel_array exactly for {sum(round_trips)} of {len(round_trips)} encodings")

In [ ]:
shifted_f = 1.02 * letter_f(y_true, x_true, shift_x=3.0)[None] * profile_z[:, None, None]
reference = encode("HFP", axes_true, pixels_true)
evaluation = encode("HFDR", axes_true, np.rint(shifted_f / DOSE_GRID_SCALING).astype(np.uint32))

axes_reference, dose_reference = pymedphys.dicom.zyx_and_dose_from_dataset(reference)
axes_evaluation, dose_evaluation = pymedphys.dicom.zyx_and_dose_from_dataset(evaluation)
gamma = pymedphys.gamma(axes_reference, dose_reference, axes_evaluation, dose_evaluation, 3, 2, lower_percent_dose_cutoff=10)
z_ref, y_ref, x_ref = axes_reference

# Choose the plane by position: z = 21 mm.
k = int(np.argmin(np.abs(z_ref - 21.0)))
k_stored = int(np.argmin(np.abs(voxel_positions(reference)[:, 0, 0, 2] - 21.0)))
gamma_stored = to_storage_order(gamma, reference)
stored_dose = reference.pixel_array * DOSE_GRID_SCALING
levels = [0.25, 1.0, 1.75]

fig, axs = plt.subplots(2, 2, figsize=(11, 10), layout="constrained")
ax = axs[0, 0]
ax.imshow(dose_reference[k], cmap=DOSE_CMAP, vmin=0, vmax=2, extent=image_extent(x_ref, y_ref))
ax.contour(x_ref, y_ref, dose_reference[k], levels=levels, colors=TRUTH, linewidths=0.8)
ax.set_title("reference dose on its axes (from the conversion)")
ax = axs[0, 1]
mesh = ax.pcolormesh(x_ref, y_ref, gamma[k], shading="nearest", cmap=GAMMA_CMAP, vmin=0, vmax=2)
ax.contour(x_ref, y_ref, dose_reference[k], levels=levels, colors=TRUTH, linewidths=0.8)
ax.set_ylim(y_ref[-1] + 1, y_ref[0] - 1)
ax.set_title("correct: gamma on the reference axes, with reference isodoses")
fig.colorbar(mesh, ax=axs[0, 1], label="γ (3%/2 mm)", shrink=0.8)
for ax in axs[0]:
    ax.set_aspect("equal")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
for ax, overlay, title in (
    (axs[1, 0], gamma[k], "incorrect: gamma drawn on pixel_array by index"),
    (axs[1, 1], gamma_stored[k_stored], "correct: gamma rearranged with to_storage_order"),
):
    ax.imshow(stored_dose[k_stored], cmap=DOSE_CMAP, vmin=0, vmax=2)
    ax.contour(np.nan_to_num(overlay), levels=[1.0], colors="#e34948", linewidths=1.5)
    ax.set_title(title)
    ax.set_xlabel("stored column index")
    ax.set_ylabel("stored row index")
fig.suptitle("Head first prone reference, z = 21 mm; red lines enclose γ > 1")
plt.show()

In the top row, gamma and the reference dose share the reference axes, and the failing regions follow the edges of the F that run along y, where the 3 mm shift along x matters. In the bottom left, the same gamma array is drawn over the stored pixel data by index: for head first prone the stored rows and columns both run backwards, so the gamma pattern is rotated by 180° relative to the dose. For decubitus orientations the stored array is also transposed and can have a different shape.

In practice:

- keep each axis tuple with its own array, and index gamma with the reference axes;
- choose planes by position, for example `k = np.argmin(np.abs(z_ref - z_target))`, because the evaluation grid has its own axes and a shared index need not mean a shared position;
- subtract doses only where their sample positions coincide, or after interpolating onto a common grid;
- to overlay gamma on `pixel_array`, rearrange gamma into stored order, as `to_storage_order` does; and
- treat `invert_yaxis` and similar calls as display choices: they change the view, not the positions of the voxels.

## 6. Speed

The changes affect three parts of the search loop:

1. The default interpolator receives a reshaped view of the search points instead of a column-stacked copy made for every shell and memory chunk. The evaluation grid is validated and converted to contiguous arrays once, when gamma starts.
2. The SciPy interpolator is built once per gamma calculation instead of once per shell and chunk.
3. The search starts at the smallest distance between the grids, and it stops at the largest distance between them if that is reached before `max_gamma` times the distance threshold. For overlapping grids, as in these benchmarks, the search still starts at zero, and the upper limit matters only for points that never find a gamma value.

The previous and current implementations were timed on the same inputs, alternately and in fresh processes, with the code below. The cell is not run when the documentation is built.

In [ ]:
# Times the previous and current gamma implementations on identical inputs.
# Set the two paths to the lib directories of two checkouts, for example main
# before this change (866f83e) and this change, then run this cell.
import json
import os
import subprocess
import sys
import tempfile

PREVIOUS_LIB = "/path/to/previous/lib"
CURRENT_LIB = "/path/to/current/lib"
CASES = ["3d", "3d-maxgamma", "3d-local", "3d-scipy", "2d", "2d-scipy"]
ROUNDS, REPEATS = 3, 2

WORKER = r'''
"""Time pymedphys.gamma on fixed synthetic cases; run once per source tree.

Usage: python bench_worker.py CASE REPEATS
Prints one JSON line: {"case", "times", "gamma_sum", "nan_count", "file"}.
"""

import json
import sys
import time
import warnings

import numpy as np
import scipy.ndimage

import pymedphys


def field(axes, centre_shift=(0.0, 0.0, 0.0), scale=1.0):
    """Smooth, clinically shaped dose: two crossing fields and a boost."""
    grids = np.meshgrid(*axes, indexing="ij")
    ndim = len(axes)
    dose = np.zeros(grids[0].shape)

    def box(half_widths, centre, weight):
        inside = np.ones(grids[0].shape, dtype=bool)
        for g, h, c, s in zip(grids, half_widths, centre, centre_shift):
            inside &= np.abs(g - c - s) <= h
        return weight * inside

    centre = [0.0] * ndim
    dose += box([40, 60, 25][:ndim], centre, 1.0)
    dose += box([55, 30, 45][:ndim], centre, 0.8)
    dose += box([15, 15, 15][:ndim], [5, -8, 4][:ndim], 0.6)
    spacing = [a[1] - a[0] for a in axes]
    sigma = [6.0 / s for s in spacing]
    dose = scipy.ndimage.gaussian_filter(dose, sigma, mode="constant")
    return 2.0 * scale * dose / dose.max()


def case_inputs(case):
    if case.startswith("3d"):
        ref_axes = tuple(np.arange(-n, n + 1e-9, 2.5) for n in (50.0, 70.0, 70.0))
        eval_axes = ref_axes
    elif case.startswith("2d"):
        ref_axes = tuple(np.arange(-n, n + 1e-9, 0.5) for n in (100.0, 100.0))
        eval_axes = ref_axes
    else:
        raise ValueError(case)
    ndim = len(ref_axes)
    ref = field(ref_axes)
    ev = field(eval_axes, centre_shift=(1.0, -0.7, 0.5)[:ndim], scale=1.02)
    options = dict(
        dose_percent_threshold=3,
        distance_mm_threshold=3,
        lower_percent_dose_cutoff=10,
    )
    if "scipy" in case:
        options["interp_algo"] = "scipy"
    if "local" in case:
        options.update(dose_percent_threshold=2, distance_mm_threshold=2, local_gamma=True)
    if "maxgamma" in case:
        options["max_gamma"] = 2
    return ref_axes, ref, eval_axes, ev, options


def main():
    case, repeats = sys.argv[1], int(sys.argv[2])
    ref_axes, ref, eval_axes, ev, options = case_inputs(case)
    warnings.simplefilter("ignore")
    # Warm-up: loads cached Numba kernels and SciPy code paths.
    gamma = pymedphys.gamma(ref_axes, ref, eval_axes, ev, **options)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        gamma = pymedphys.gamma(ref_axes, ref, eval_axes, ev, **options)
        times.append(time.perf_counter() - start)
    print(
        json.dumps(
            {
                "case": case,
                "times": times,
                "gamma_sum": float(np.nansum(gamma)),
                "nan_count": int(np.isnan(gamma).sum()),
                "points": int(np.size(gamma)),
                "file": pymedphys.__file__,
            }
        )
    )


if __name__ == "__main__":
    main()
'''

with tempfile.TemporaryDirectory() as directory:
    worker = os.path.join(directory, "worker.py")
    with open(worker, "w", encoding="utf-8") as file:
        file.write(WORKER)
    timings = {case: {"previous": [], "current": []} for case in CASES}
    for round_index in range(ROUNDS):
        for case in CASES:
            order = ["previous", "current"] if round_index % 2 == 0 else ["current", "previous"]
            for version in order:
                library = PREVIOUS_LIB if version == "previous" else CURRENT_LIB
                result = subprocess.run(
                    [sys.executable, worker, case, str(REPEATS)],
                    env=dict(os.environ, PYTHONPATH=library),
                    cwd=directory, capture_output=True, text=True, check=True,
                )
                timings[case][version] += json.loads(result.stdout)["times"]
print(json.dumps(timings, indent=1))

In [ ]:
# Recorded on 26 September 2026: Linux, 4-core Intel Xeon at 2.8 GHz, Numba's
# default of 4 threads, Python 3.12.3, NumPy 1.26.4, SciPy 1.17.1, Numba 0.67.0.
# Previous: main at 866f83e, whose gamma code matches 0.41.0. Current: this change.
# Three rounds alternating the two versions, two timed runs each after a warm-up.
# 3-D: 41 x 57 x 57 points at 2.5 mm; 2-D: 401 x 401 points at 0.5 mm; 10% cutoff.
RECORDED = {
    "times": {
        "3d": {
            "previous": [10.858, 11.44, 14.624, 14.187, 14.224, 13.731],
            "current": [7.285, 7.916, 7.269, 10.298, 8.452, 6.237],
        },
        "3d-maxgamma": {
            "previous": [13.154, 14.827, 14.304, 12.58, 16.654, 12.628],
            "current": [11.258, 6.553, 7.204, 6.729, 7.985, 8.507],
        },
        "3d-local": {
            "previous": [55.212, 53.798, 59.327, 58.085, 58.399, 65.482],
            "current": [37.245, 39.76, 34.032, 37.449, 34.818, 38.931],
        },
        "3d-scipy": {
            "previous": [42.715, 47.628, 43.2, 42.175, 42.696, 41.175],
            "current": [42.339, 41.677, 42.52, 42.518, 42.225, 41.969],
        },
        "2d": {
            "previous": [0.211, 0.232, 0.222, 0.21, 0.198, 0.191],
            "current": [0.237, 0.191, 0.215, 0.185, 0.187, 0.188],
        },
        "2d-scipy": {
            "previous": [0.39, 0.422, 0.371, 0.373, 0.349, 0.363],
            "current": [0.375, 0.356, 0.366, 0.349, 0.438, 0.385],
        },
    },
}

labels = {
    "3d": "3-D, 3%/3 mm\ndefault settings",
    "3d-maxgamma": "3-D, 3%/3 mm\nmax_gamma = 2",
    "3d-local": "3-D, 2%/2 mm\nlocal",
    "3d-scipy": "3-D, 3%/3 mm\nSciPy interpolator",
    "2d": "2-D, 3%/3 mm\ndefault settings",
    "2d-scipy": "2-D, 3%/3 mm\nSciPy interpolator",
}
print(f"{'case':<32}{'previous (s)':>14}{'current (s)':>13}{'ratio':>8}")
ratios = {}
for case, times in RECORDED["times"].items():
    previous, current = np.median(times["previous"]), np.median(times["current"])
    ratios[case] = current / previous
    print(f"{labels[case].replace(chr(10), ', '):<32}{previous:>14.2f}{current:>13.2f}{ratios[case]:>8.2f}")

fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
positions = np.arange(len(ratios))
for position, case in zip(positions, ratios):
    times = RECORDED["times"][case]
    each = np.array(times["current"])[:, None] / np.array(times["previous"])[None, :]
    ax.plot([position, position], [each.min(), each.max()], color=CURRENT, linewidth=2, alpha=0.35)
    ax.plot(position, ratios[case], "o", color=CURRENT, markersize=8, markeredgecolor="#fcfcfb", markeredgewidth=2)
    ax.text(position + 0.12, ratios[case], f"{ratios[case]:.2f}", va="center", fontsize=9)
ax.axhline(1.0, color=MUTED, linewidth=1)
ax.text(-0.4, 1.01, "previous speed", ha="left", va="bottom", color=SECONDARY, fontsize=8.5)
ax.set_xticks(positions, [labels[case] for case in ratios])
ax.set_ylim(0.3, 1.35)
ax.set_ylabel("current time / previous time")
ax.grid(axis="y")
ax.set_title("Gamma run time relative to the previous implementation: median, with the range of all pairings")
plt.show()

With the default interpolator, the median three-dimensional run time fell by 36% to 46%, and every current run was faster than every previous run of the same case. With the SciPy interpolator, and in two dimensions, where each calculation takes a fraction of a second, the medians differ by at most 10% and the ranges overlap, so no change is measurable. Building the SciPy interpolator was cheap compared with evaluating it at every search point, so building it once saves little.

Every gamma value was identical: in all six cases, the previous and current implementations returned element-wise equal arrays, including the positions of unevaluated points.

The reordering itself costs nothing measurable. The cell below times the current implementation on a smaller grid with the evaluation axes ascending and with all three reversed, on the documentation build host.

In [ ]:
live_axes = (-20.0 + 2.5 * np.arange(17), -40.0 + 2.5 * np.arange(33), -40.0 + 2.5 * np.arange(33))
live_reference = box_field(*live_axes, centre_x=0.0)
live_evaluation = 1.02 * box_field(*live_axes, centre_x=1.0)
reversed_axes = tuple(axis[::-1] for axis in live_axes)
reversed_evaluation = live_evaluation[::-1, ::-1, ::-1]


def best_time(axes_evaluation, dose_evaluation, repeats=2):
    times, result = [], None
    for _ in range(repeats + 1):  # the first call also compiles or loads the Numba kernels
        start = time.perf_counter()
        result = pymedphys.gamma(live_axes, live_reference, axes_evaluation, dose_evaluation, 3, 3, lower_percent_dose_cutoff=10)
        times.append(time.perf_counter() - start)
    return min(times[1:]), result


ascending_time, ascending_gamma = best_time(live_axes, live_evaluation)
reversed_time, reversed_gamma = best_time(reversed_axes, reversed_evaluation)
print(f"evaluation axes ascending: {ascending_time:.2f} s; reversed: {reversed_time:.2f} s")
print("identical gamma:", np.array_equal(ascending_gamma, reversed_gamma, equal_nan=True))

start = time.perf_counter()
for _ in range(20):
    _prepare_evaluation_grid(reversed_axes, reversed_evaluation, "pymedphys")
print(f"preparing the reversed evaluation grid takes {(time.perf_counter() - start) / 20 * 1e3:.2f} ms")

## 7. Was an earlier result affected?

The previous displacement of every voxel follows from the geometry attributes alone, so it can be calculated for files that were compared with earlier versions, without their pixel data; read them with `pydicom.dcmread(path, stop_before_pixels=True)`. For a comparison between two grids, the relative displacement matters. Comparisons with anything else in patient coordinates, such as structures, points, or profiles, are affected by the absolute displacement of each grid.

In [ ]:
def format_mm(vector):
    return "(" + ", ".join(f"{value + 0.0:+.1f}" for value in vector) + ") mm"


def describe_previous_registration(reference, evaluation):
    """Print how the previous conversion placed a reference and an evaluation grid."""
    translations = {}
    for role, ds in (("reference", reference), ("evaluation", evaluation)):
        try:
            displacement = legacy_displacement(ds).reshape(-1, 3)
        except (AttributeError, ValueError) as error:
            print(f"  {role}: the previous conversion failed: {error}")
            continue
        if np.allclose(displacement, displacement[0], rtol=0, atol=1e-6):
            translations[role] = displacement[0]
            print(f"  {role}: translated by {format_mm(displacement[0])}")
        else:
            largest = np.linalg.norm(displacement, axis=1).max()
            print(f"  {role}: transposed in the transverse plane, voxels displaced by up to {largest:.1f} mm")
    if len(translations) == 2:
        relative = translations["evaluation"] - translations["reference"]
        print(f"  evaluation relative to reference: {format_mm(relative)}, {np.linalg.norm(relative):.1f} mm in total")
    try:
        descending = any(axis.size > 1 and axis[1] < axis[0] for axis in legacy_xyz_axes_from_dataset(evaluation))
    except AttributeError:
        descending = False
    if descending:
        print(
            "  the previous evaluation axes descended: with the default interpolator of 0.41.0, "
            "gamma raised an error, returned NaN, or did not terminate"
        )


examples = {
    "HFS grid and its crop": (encode("HFS", box_axes, box_pixels), encode("HFS", (bz, by, bx[:-4]), box_pixels[..., :-4])),
    "HFP grid and its crop": (full, cropped),
    "HFP reference, HFS evaluation": (full, encode("HFS", box_axes, box_pixels)),
    "HFDL reference, HFS evaluation": (encoded["HFDL"], encoded["HFS"]),
}
for label, (reference_ds, evaluation_ds) in examples.items():
    print(label)
    describe_previous_registration(reference_ds, evaluation_ds)

The head first supine pair was unaffected. The head first prone pair was misregistered by the 10 mm crop, although both grids were stored the same way, and gamma with this evaluation grid needed `interp_algo="scipy"` to run at all. The head first prone reference against a head first supine evaluation gave results without an error, 40 mm out of register. The decubitus reference was transposed.

## 8. Remaining limitations

- Oblique orientations are rejected (section 3.6).
- The shell search can step past an evaluation grid that is narrower than one search step, or that has a singleton axis, and report NaN or an inaccurate value; see [#2070](https://github.com/pymedphys/pymedphys/issues/2070). Compare planes with two-dimensional arrays (section 3.5).
- Two private helpers, `get_dose_grid_structure_mask` and `DicomDose.coords`, still assume that stored rows run along $y$, which is wrong for decubitus grids. The public conversion and interpolation functions do not use them.
- The IEC patient option of the private `xyz_axes_from_dataset` raises `NotImplementedError`; its IEC fixed option keeps the previous convention.